# Notebook 6 — Fine-tuning Pretrained GloVe Embeddings

*Module 2 (Text Similarity) — ML & NLP by Data Trainers LLC.*

## The Situation

In **Notebook 5** you trained a CBOW model from scratch on 5,000 Yelp reviews. You got vectors that knew **pizza** is similar to **burger**, but they had no clue about **king**, **queen**, **paris**, **france** — those words barely appeared in restaurant reviews. Your embeddings were *domain-good but globally-weak*.

Meanwhile, Stanford's NLP group has already trained vectors on **6 billion tokens** of Wikipedia + Gigaword. Those vectors know everything: capital cities, gender analogies, verb tenses, whatever you throw at them. They are public. They are free. They are a single text file away.

## The Task

Stop training from scratch. Use **GloVe** as the starting point and apply three classic recipes:

1. **Variant A — random init** (the NB5 baseline, for comparison).
2. **Variant B — frozen GloVe**: load GloVe into the embedding layer and lock it (`freeze=True`). Only the prediction head trains.
3. **Variant C — fine-tuned GloVe**: load GloVe but allow gradients to flow through it (`freeze=False`). The vectors shift toward the Yelp domain.

## The Action

Download GloVe 6B 100d, build an *aligned* `(vocab_size, 100)` embedding matrix where row `i` is the GloVe vector for word `i` of the Yelp vocab, then wire it into `nn.Embedding.from_pretrained(...)`. Train all three variants for 5 epochs with identical hyperparameters and compare.

## The Result

By the end of this notebook you will know:

- That **transfer learning works in NLP** the same way it works in vision (use a pretrained backbone).
- The PyTorch idiom `nn.Embedding.from_pretrained(matrix, freeze=True/False, padding_idx=0)`.
- When to freeze (small dataset, generic task) vs. fine-tune (large dataset, domain-specific task).
- How `most_similar('cheap')` shifts from the GloVe meaning (*inexpensive, affordable*) to the Yelp meaning (*crappy, stale*) when you fine-tune.

**Runtime**: ~45 minutes on Colab free tier (most of it is the GloVe download).

**Prerequisites**: Notebook 5 (CBOW). We reuse the model class and the CBOW data generator.

**Framework**: PyTorch.

## Section 0: Environment Setup

This notebook is written for Google Colab (free tier, GPU optional). Run the next cell once per session to install the packages.

In [ ]:
# Install required packages (skip if you already have them locally)
!pip install -q torch scikit-learn pandas numpy matplotlib gensim textblob

In [ ]:
# Core imports
import os, re, random, math, time
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import gensim
from textblob import TextBlob

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"PyTorch version: {torch.__version__}")
print("Environment ready.")

In [ ]:
# Hyperparameters — keep these at the top so they are easy to change.
EMBEDDING_DIM = 100        # MUST match the GloVe file we download (100d)
CORPUS_SIZE   = 5000       # number of Yelp reviews we'll train on
WINDOW_SIZE   = 2          # CBOW context window: 2 words on each side
BATCH_SIZE    = 128
EPOCHS        = 5
LR_FROZEN     = 1e-3       # only the Linear head updates -> can use a higher LR
LR_FINETUNE   = 5e-4       # lower LR so pretrained vectors don't drift too fast
MIN_FREQ      = 2          # vocabulary cutoff: drop words seen <2 times
PAD_IDX       = 0          # convention: id 0 is the padding token

print(f"EMBEDDING_DIM={EMBEDDING_DIM}  CORPUS_SIZE={CORPUS_SIZE}  WINDOW_SIZE={WINDOW_SIZE}")
print(f"BATCH_SIZE={BATCH_SIZE}  EPOCHS={EPOCHS}  LR_FROZEN={LR_FROZEN}  LR_FINETUNE={LR_FINETUNE}")

## Section 1: Transfer Learning for NLP

Imagine training **ResNet-50** from scratch every time you want to classify a new image dataset. That would be insane — everyone uses a pretrained ImageNet checkpoint and either freezes the backbone or fine-tunes a few layers. The same idea applies to NLP.

**Pretrained word embeddings** are the NLP equivalent of an ImageNet backbone: someone (Stanford, Google, Facebook) trained vectors on a massive corpus you couldn't reasonably touch on your laptop, and you get to download the result for free.

### A short history

| Year | Model | Idea |
|------|-------|------|
| 2013 | **Word2Vec** (Mikolov et al., Google) | Predict context words from center word (skip-gram) or center from context (CBOW). |
| 2014 | **GloVe** (Pennington et al., Stanford) | Factorize a global word-word **co-occurrence** matrix. |
| 2018+ | ELMo, BERT, GPT | Contextual embeddings — the vector for *bank* depends on the sentence. Coming in Module 3. |

We'll use **GloVe** today because (a) the file format is dead simple — `word v1 v2 ... v100` per line — and (b) it's small enough (~330MB for 100d) to download in a Colab session.

### Why we said *transfer*

Transfer learning works because language has universal patterns: **king is to man as queen is to woman**, **paris is to france as rome is to italy**. Wikipedia + Gigaword saw billions of examples of these patterns. Your Yelp corpus saw none. So instead of asking your CBOW model to learn English from 5,000 reviews, hand it the GloVe vectors as initialization and let it specialize from there.

In [ ]:
# Tiny visual: in NB5 we started from white noise. Today we start from GloVe.
# (Just an illustration — no real GloVe loaded yet.)
fig, axes = plt.subplots(1, 2, figsize=(8, 2.6))
axes[0].imshow(np.random.randn(20, EMBEDDING_DIM), cmap='RdBu', aspect='auto')
axes[0].set_title('NB5: random init (white noise)')
axes[0].set_xlabel('embedding dim'); axes[0].set_ylabel('word id')
axes[1].imshow(np.random.randn(20, EMBEDDING_DIM) * 0.4, cmap='RdBu', aspect='auto')
axes[1].set_title('NB6: GloVe init (structured)')
axes[1].set_xlabel('embedding dim')
plt.tight_layout(); plt.show()

## Section 2: Loading GloVe

GloVe ships as a plain text file. Each line is one word followed by 100 floats:

```
the 0.418 0.24968 -0.41242 ...   (100 numbers)
of  0.70853 0.57088 -0.4716 ...
and 0.43181 -0.21997 0.18353 ...
```

We need a Python `dict` mapping each word to a `numpy` array of shape `(100,)`. Then we can look up `glove['king']`, do arithmetic, find nearest neighbors — the whole bag of tricks.

### Step 1: download the file

GloVe 6B 100d is hosted on a Dropbox mirror for the course. The file is ~330MB so the download takes ~1 minute on Colab. We cache it locally — if you re-run the cell, it skips the download.

In [ ]:
# Download GloVe 6B 100d (~330MB). Cached after the first run.
GLOVE_PATH = './glove.6B.100d.txt'
if not os.path.exists(GLOVE_PATH):
    !wget -q -O ./glove.6B.100d.txt 'https://www.dropbox.com/s/dl1vswq2sz5f1ws/glove.6B.100d.txt?dl=1'
print(f"GloVe file size: {os.path.getsize(GLOVE_PATH) / 1e6:.1f} MB")

# Peek at the first 2 lines so you can see the file format with your own eyes
with open(GLOVE_PATH, 'r', encoding='utf-8') as f:
    for _ in range(2):
        line = f.readline()
        word, rest = line.split(maxsplit=1)
        print(f"word={word!r}   first 5 dims=[{', '.join(rest.split()[:5])} ...]")

### Demo: parse the GloVe file into a Python dict

Two performance tricks worth knowing:

- `line.split(maxsplit=1)` — splits the line into exactly 2 pieces: the word, and a single string of all the numbers. Faster than splitting on every space.
- `np.fromstring(rest, dtype='float32', sep=' ')` — parses the numeric string straight into a `float32` array. (Use `float32` not `float64` — half the memory, and PyTorch wants float32 anyway.)

Loading 400K vectors takes ~30 seconds.

In [ ]:
def load_glove(path):
    """Parse a GloVe text file into {word: np.float32 array of shape (D,)}."""
    glove = {}
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            word, rest = line.split(maxsplit=1)
            # 'rest' is a single string of space-separated floats
            glove[word] = np.fromstring(rest, dtype='float32', sep=' ')
    return glove

t0 = time.time()
glove = load_glove(GLOVE_PATH)
print(f"Loaded {len(glove):,} word vectors in {time.time() - t0:.1f}s")
print(f"Each vector has shape: {glove['king'].shape} (dtype={glove['king'].dtype})")

In [ ]:
# Sanity check: 'king' and 'queen' should look reasonable and have similar magnitudes.
print("glove['king'][:8]  =", np.round(glove['king'][:8], 3))
print("glove['queen'][:8] =", np.round(glove['queen'][:8], 3))
print(f"\n||king||  = {np.linalg.norm(glove['king']):.3f}")
print(f"||queen|| = {np.linalg.norm(glove['queen']):.3f}")

# Cosine similarity between king and queen
k, q = glove['king'], glove['queen']
cos_kq = np.dot(k, q) / (np.linalg.norm(k) * np.linalg.norm(q))
print(f"\ncosine(king, queen) = {cos_kq:.3f}   <-- close to 1 means very similar")

### Demo: the famous analogy

*king − man + woman ≈ ?*  The vector arithmetic should land near *queen*. This is the classic sanity check that GloVe is loaded correctly.

We'll write a tiny `nearest_neighbors` helper that takes a target vector and returns the top-N closest words from the GloVe dict by cosine similarity.

In [ ]:
# We pre-stack the GloVe matrix once for fast batched cosine similarity.
GLOVE_WORDS = list(glove.keys())
GLOVE_MATRIX = np.stack([glove[w] for w in GLOVE_WORDS])           # (400000, 100)
GLOVE_NORMS = np.linalg.norm(GLOVE_MATRIX, axis=1, keepdims=True)  # (400000, 1)

def nearest_neighbors(vec, topn=5, exclude=()):
    """Return the top-N closest GloVe words to `vec` by cosine similarity."""
    vec = vec.astype('float32')
    vec_norm = np.linalg.norm(vec) + 1e-9
    sims = (GLOVE_MATRIX @ vec) / (GLOVE_NORMS.squeeze() * vec_norm)
    # Get more than topn so we can drop excluded words and still have enough left
    top_idx = np.argpartition(-sims, topn + len(exclude))[:topn + len(exclude)]
    top_idx = top_idx[np.argsort(-sims[top_idx])]  # sort the small slice exactly
    out = []
    for i in top_idx:
        w = GLOVE_WORDS[i]
        if w in exclude:
            continue
        out.append((w, float(sims[i])))
        if len(out) == topn:
            break
    return out

# The famous analogy: king - man + woman -> ?
analogy_vec = glove['king'] - glove['man'] + glove['woman']
print("king - man + woman -> nearest:")
for w, s in nearest_neighbors(analogy_vec, topn=5, exclude={'king', 'man', 'woman'}):
    print(f"   {w:<15s} sim={s:.3f}")

### Lab 2.1: Country–Capital analogy

GloVe captures lots of relational patterns. One is *country → capital*: the vector difference `paris - france` is approximately the same as `rome - italy`. Verify this for yourself.

**Your task**: compute the analogy `paris - france + italy` and print the top-5 nearest words (excluding the three input words). You should see *rome* near the top.

Hints:

- All four words are lowercase in GloVe — `glove['paris']`, not `glove['Paris']`.
- Pass `exclude={'paris', 'france', 'italy'}` to `nearest_neighbors` so the input words don't dominate the result.

In [ ]:
# Lab 2.1: country-capital analogy
# 1. Compute the analogy vector: paris - france + italy
analogy = None  # YOUR CODE

# 2. Get the top-5 nearest neighbors, excluding the input words
neighbors = None  # YOUR CODE

# Verification
if neighbors is not None:
    print("paris - france + italy -> nearest:")
    for w, s in neighbors:
        print(f"   {w:<15s} sim={s:.3f}")

## Section 3: Building the Aligned Embedding Matrix

GloVe gives us a `dict[word -> vector]`. PyTorch wants a single tensor of shape `(vocab_size, embedding_dim)`. We need to bridge the two.

### The alignment problem

Our Yelp vocabulary is something like:

```
vocab = {'<pad>': 0, '<unk>': 1, 'the': 2, 'pizza': 3, 'yum': 4, 'mediocre': 5, ...}
```

Some of these words are in GloVe (`the`, `pizza`, `mediocre`), some aren't (`yum`, our two special tokens). We need to build a matrix where:

- **Row 0** (`<pad>`) is **all zeros** — padding must contribute nothing to the mean.
- **Row 1** (`<unk>`) gets a small random vector (avoid zeros so it's distinguishable from padding).
- **Other rows** get either the GloVe vector for that word or a small random vector if the word is OOV (out-of-GloVe).

> ⚠️ **Why small random and not zero for OOV?** Identical zero rows would mean every OOV word is *the same word* to the model. Initializing with `np.random.normal(0, 0.1, ...)` breaks symmetry and lets the model differentiate them as it trains.

Let's first build the Yelp corpus and vocabulary.

In [ ]:
# Download the Yelp reviews corpus (small, ~25MB)
YELP_PATH = './yelp.csv'
if not os.path.exists(YELP_PATH):
    !wget -q -O ./yelp.csv 'https://www.dropbox.com/s/xds4lua69b7okw8/yelp.csv?dl=1'
yelp = pd.read_csv(YELP_PATH)
# Keep only 5-star and 1-star reviews to match NB5 (extreme sentiments)
yelp_extreme = yelp[(yelp.stars == 5) | (yelp.stars == 1)]
print(f"Total Yelp 1/5 star reviews: {len(yelp_extreme):,}")

# Tokenize using TextBlob (same as NB5). Keep first CORPUS_SIZE reviews with > 3 words.
def tokenize(text):
    return [w.lower() for w in TextBlob(text).words]

raw_texts = yelp_extreme['text'].astype(str).tolist()
corpus_tokens = []
for txt in raw_texts:
    toks = tokenize(txt)
    if len(toks) > 3:
        corpus_tokens.append(toks)
    if len(corpus_tokens) >= CORPUS_SIZE:
        break
print(f"Corpus size: {len(corpus_tokens):,} reviews")
print(f"First review (first 12 tokens): {corpus_tokens[0][:12]}")

In [ ]:
# Build the vocabulary: <pad>=0, <unk>=1, then words by frequency (>= MIN_FREQ).
counter = Counter()
for toks in corpus_tokens:
    counter.update(toks)

PAD_TOKEN, UNK_TOKEN = '<pad>', '<unk>'
vocab = {PAD_TOKEN: 0, UNK_TOKEN: 1}
for tok, freq in counter.most_common():
    if freq >= MIN_FREQ:
        vocab[tok] = len(vocab)
vocab_size = len(vocab)
id_to_word = {i: w for w, i in vocab.items()}
print(f"Vocabulary size (incl. <pad> and <unk>): {vocab_size:,}")
print(f"Sample first 10 entries: {list(vocab.items())[:10]}")

### Demo: build the aligned embedding matrix and report coverage

Walk over each word in our vocabulary in id order (0, 1, 2, ...). For each one, look it up in GloVe; if found, place that vector at row `i`; otherwise, leave it as the small random initialization. Report how many words were covered.

We'll write the function inline for the demo, then ask you to re-implement a clean version in the lab.

In [ ]:
# Demo build (we re-implement this cleanly in the lab below)
rng = np.random.RandomState(SEED)
demo_matrix = rng.normal(0, 0.1, size=(vocab_size, EMBEDDING_DIM)).astype('float32')
demo_matrix[PAD_IDX] = 0.0  # padding row must be zeros (see padding_idx pitfall)

hits, misses = 0, 0
for word, idx in vocab.items():
    if word in (PAD_TOKEN, UNK_TOKEN):
        continue
    vec = glove.get(word)
    if vec is not None:
        demo_matrix[idx] = vec
        hits += 1
    else:
        misses += 1
coverage = hits / (hits + misses)
print(f"GloVe coverage: {hits:,}/{hits + misses:,} = {coverage:.1%}")
print(f"OOV (kept as small random): {misses:,} words")
print(f"Embedding matrix shape: {demo_matrix.shape}, dtype: {demo_matrix.dtype}")
print(f"Row 0 (PAD) is all zero? {np.all(demo_matrix[0] == 0)}")

### Lab 3.1: Implement `build_embedding_matrix`

Wrap the demo logic into a clean reusable function with this signature:

```python
def build_embedding_matrix(vocab, glove, embedding_dim, pad_idx=0, seed=42):
    """Returns a (vocab_size, embedding_dim) float32 numpy matrix.
    Row pad_idx is all zeros; words present in GloVe get their GloVe vector;
    OOV words get small random init from N(0, 0.1).
    Also returns (hits, misses) for reporting.
    """
```

Hints:

- Initialize with `np.random.RandomState(seed).normal(0, 0.1, size=(V, D)).astype('float32')`.
- Then `matrix[pad_idx] = 0`.
- Loop `for word, idx in vocab.items():` and overwrite rows where `glove.get(word)` is not `None`.
- Skip the `pad_idx` row when looking up — it stays zero.

In [ ]:
# Lab 3.1: implement the function
def build_embedding_matrix(vocab, glove, embedding_dim, pad_idx=0, seed=42):
    # 1. Allocate a (vocab_size, embedding_dim) float32 matrix from N(0, 0.1)
    matrix = None  # YOUR CODE

    # 2. Set the padding row to zeros
    # YOUR CODE

    # 3. Loop over vocab; overwrite rows for words present in glove
    hits, misses = 0, 0
    # YOUR CODE

    return matrix, hits, misses

# Build it and verify
embedding_matrix, hits, misses = build_embedding_matrix(vocab, glove, EMBEDDING_DIM, pad_idx=PAD_IDX, seed=SEED)
if embedding_matrix is not None:
    print(f"Shape: {embedding_matrix.shape}, dtype: {embedding_matrix.dtype}")
    print(f"Coverage: {hits} / {hits + misses} = {hits / max(1, hits + misses):.1%}")
    print(f"Row 0 zero? {np.all(embedding_matrix[0] == 0)}")

    # Convert numpy -> torch (float32). This is the tensor we hand to nn.Embedding.
    embedding_matrix_t = torch.FloatTensor(embedding_matrix)
    print(f"Torch tensor shape: {embedding_matrix_t.shape}, dtype: {embedding_matrix_t.dtype}")

## Section 4: Three CBOW Variants

We are going to train **the exact same CBOW architecture** three times, changing only how the embedding layer is initialized:

| Variant | Embedding init | `freeze` |
|---------|----------------|----------|
| **A** Random | `nn.Embedding(V, D)` (default Xavier-ish) | n/a (trainable) |
| **B** Frozen GloVe | `nn.Embedding.from_pretrained(matrix, freeze=True)` | `True` |
| **C** Fine-tuned GloVe | `nn.Embedding.from_pretrained(matrix, freeze=False)` | `False` |

All three share the same data, same window size, same batch size, same number of epochs. Only the embedding initialization changes. This is a clean **A/B/C test** — any difference in the loss curves comes from the embedding init.

### The PyTorch idiom

```python
embedding_matrix_t = torch.FloatTensor(embedding_matrix)  # (V, D), float32

# Frozen: weights do NOT update
emb_frozen = nn.Embedding.from_pretrained(embedding_matrix_t, freeze=True, padding_idx=0)

# Fine-tune: weights DO update via backprop
emb_finetune = nn.Embedding.from_pretrained(embedding_matrix_t, freeze=False, padding_idx=0)
```

Translation from the old Keras code:

| Keras | PyTorch |
|-------|---------|
| `Embedding(V, D, embeddings_initializer=Constant(m), trainable=False)` | `nn.Embedding.from_pretrained(m, freeze=True)` |
| `Embedding(V, D, embeddings_initializer=Constant(m), trainable=True)` | `nn.Embedding.from_pretrained(m, freeze=False)` |
| `Embedding(..., mask_zero=True)` | `padding_idx=0` |

In [ ]:
# The CBOW model. Same as NB5 except __init__ takes an optional pretrained matrix.
class CBOW(nn.Module):
    def __init__(self, vocab_size, embedding_dim, pretrained_matrix=None, freeze=True, padding_idx=PAD_IDX):
        super().__init__()
        if pretrained_matrix is None:
            # Variant A: random init
            self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=padding_idx)
            # Make sure the pad row stays zero even after random init
            with torch.no_grad():
                self.embedding.weight[padding_idx].zero_()
        else:
            # Variants B and C: load pretrained GloVe, freeze (B) or not (C)
            self.embedding = nn.Embedding.from_pretrained(
                pretrained_matrix, freeze=freeze, padding_idx=padding_idx
            )
        self.linear = nn.Linear(embedding_dim, vocab_size)

    def forward(self, context_ids):
        # context_ids: (B, 2*window)
        e = self.embedding(context_ids)        # (B, 2*window, D)
        ctx = e.mean(dim=1)                    # (B, D)  -- average context words
        return self.linear(ctx)                # (B, V)  -- raw logits (no softmax!)

# Sanity check by building the random-init variant
tmp = CBOW(vocab_size, EMBEDDING_DIM).to(device)
n = sum(p.numel() for p in tmp.parameters() if p.requires_grad)
print(f"CBOW (random init) trainable params: {n:,}")
del tmp

### CBOW data: (context, target) pairs

From NB5 — for each token in the corpus, the **context** is the surrounding `2*window` words and the **target** is the center word. We build a flat list of `(context_ids, target_id)` pairs once up front, then wrap them in a Dataset/DataLoader for batching.

In [ ]:
def build_cbow_pairs(corpus_tokens, vocab, window=WINDOW_SIZE, pad_idx=PAD_IDX, unk_idx=1):
    pairs = []
    ctx_len = 2 * window
    for toks in corpus_tokens:
        ids = [vocab.get(t, unk_idx) for t in toks]
        L = len(ids)
        for i, target in enumerate(ids):
            ctx = []
            for j in range(i - window, i + window + 1):
                if j == i or j < 0 or j >= L:
                    continue
                ctx.append(ids[j])
            # Pad to fixed length 2*window
            while len(ctx) < ctx_len:
                ctx.append(pad_idx)
            pairs.append((ctx, target))
    return pairs

pairs = build_cbow_pairs(corpus_tokens, vocab)
print(f"Total CBOW (context, target) pairs: {len(pairs):,}")
print(f"Sample pair (context_ids, target_id): {pairs[5]}")
ctx, tgt = pairs[5]
print(f"  decoded context: {[id_to_word[i] for i in ctx]}")
print(f"  decoded target : {id_to_word[tgt]}")

In [ ]:
class CBOWDataset(Dataset):
    def __init__(self, pairs):
        # Pre-stack as tensors for speed
        self.X = torch.tensor([p[0] for p in pairs], dtype=torch.long)
        self.y = torch.tensor([p[1] for p in pairs], dtype=torch.long)
    def __len__(self):
        return len(self.y)
    def __getitem__(self, i):
        return self.X[i], self.y[i]

train_ds = CBOWDataset(pairs)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
print(f"Train loader: {len(train_loader)} batches of size {BATCH_SIZE}")

### Demo: training loop helper

We'll reuse the same `train_one_variant` function for all 3 models. It:

1. Builds the optimizer (only over `requires_grad=True` params — important for the frozen variant).
2. Runs `EPOCHS` epochs of SGD on `CrossEntropyLoss`.
3. Returns the per-epoch loss list.

> ⚠️ For the **frozen** variant, `nn.Embedding.from_pretrained(..., freeze=True)` sets `requires_grad=False` on the embedding weights. Adam silently skips them — we still pass `model.parameters()`, but only the `Linear` layer actually updates.

In [ ]:
def train_one_variant(model, loader, lr, epochs=EPOCHS, label='model'):
    """Train one CBOW variant. Returns the per-epoch mean loss list."""
    # filter() so frozen params don't trigger an Adam warning on some PyTorch versions
    optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    losses = []
    for ep in range(1, epochs + 1):
        model.train()
        total, n = 0.0, 0
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            optimizer.zero_grad()
            logits = model(X)
            loss = loss_fn(logits, y)
            loss.backward()
            optimizer.step()
            total += loss.item()
            n += 1
        ep_loss = total / max(1, n)
        losses.append(ep_loss)
        print(f"[{label}] epoch {ep}/{epochs} | loss={ep_loss:.4f}")
    return losses

### Lab 4.1: Build and train the three variants

Construct each of the 3 models and train them with our helper. Use the right learning rate for each:

- Variant A (random) and Variant B (frozen GloVe) -> `LR_FROZEN` (1e-3) is fine — the embedding is either fresh or locked, so Adam can move briskly.
- Variant C (fine-tuned GloVe) -> `LR_FINETUNE` (5e-4) — the pretrained vectors are precious, we don't want to wreck them with a big update.

Hints:

- **Variant A**: `CBOW(vocab_size, EMBEDDING_DIM)` — no `pretrained_matrix`.
- **Variant B**: `CBOW(vocab_size, EMBEDDING_DIM, pretrained_matrix=embedding_matrix_t, freeze=True)`.
- **Variant C**: same as B but `freeze=False`.
- Don't forget `.to(device)` on each model.
- After construction, print `sum(p.numel() for p in model.parameters() if p.requires_grad)` so you see frozen has the smallest trainable count.

In [ ]:
# Lab 4.1: build the three variants
torch.manual_seed(SEED)  # same init randomness for fair comparison
model_random = None  # YOUR CODE: random init CBOW, .to(device)

torch.manual_seed(SEED)
model_frozen = None  # YOUR CODE: frozen GloVe CBOW, .to(device)

torch.manual_seed(SEED)
model_ft = None      # YOUR CODE: fine-tuned GloVe CBOW, .to(device)

# Verification
for name, m in [('random', model_random), ('frozen', model_frozen), ('finetune', model_ft)]:
    if m is not None:
        n_train = sum(p.numel() for p in m.parameters() if p.requires_grad)
        n_total = sum(p.numel() for p in m.parameters())
        print(f"{name:>8s} | trainable={n_train:>10,d} | total={n_total:>10,d}")

In [ ]:
# Lab 4.1 (continued): train all three variants
losses_random   = None  # YOUR CODE: train_one_variant(model_random,   train_loader, LR_FROZEN,   label='random')
losses_frozen   = None  # YOUR CODE: train_one_variant(model_frozen,   train_loader, LR_FROZEN,   label='frozen')
losses_finetune = None  # YOUR CODE: train_one_variant(model_ft,       train_loader, LR_FINETUNE, label='finetune')

### Demo: compare loss curves

We expect the **fine-tuned** variant to reach the lowest loss — it gets a head start from GloVe AND specializes to Yelp. Frozen sits in the middle (good init but inflexible). Random is the slowest learner.

In [ ]:
if losses_random and losses_frozen and losses_finetune:
    plt.figure(figsize=(7.5, 4))
    epochs_axis = list(range(1, EPOCHS + 1))
    plt.plot(epochs_axis, losses_random,   'o-', label='A: random init',     color='#FF6B6B')
    plt.plot(epochs_axis, losses_frozen,   's-', label='B: frozen GloVe',    color='#4ECDC4')
    plt.plot(epochs_axis, losses_finetune, '^-', label='C: fine-tuned GloVe', color='#5B8DEF')
    plt.xlabel('Epoch'); plt.ylabel('Cross-entropy loss')
    plt.title('CBOW: same architecture, three embedding inits')
    plt.grid(alpha=0.3); plt.legend(); plt.tight_layout(); plt.show()
else:
    print("Train the three variants in Lab 4.1 first.")

### Demo: `most_similar` per variant

We extract each model's embedding matrix, dump it to a temporary `word2vec` text file, and load it with **gensim** so we get the convenient `most_similar` API.

Watch how `most_similar('cheap')` differs across the three models. In Yelp reviews, *cheap* often means *low quality* ("the food was cheap and stale"). In GloVe, *cheap* means *inexpensive*. Fine-tuning on Yelp drags the GloVe vector toward the negative-review meaning.

In [ ]:
def model_to_keyedvectors(model, vocab, id_to_word, path):
    """Dump a trained CBOW's embedding matrix to a word2vec-format text file
    and load it back as a gensim KeyedVectors object."""
    weights = model.embedding.weight.detach().cpu().numpy()
    V, D = weights.shape
    with open(path, 'w', encoding='utf-8') as f:
        # Skip <pad> (id 0) — its all-zero row breaks gensim's normalization
        f.write(f"{V - 1} {D}\n")
        for i in range(1, V):
            w = id_to_word[i]
            vec = ' '.join(f"{x:.6f}" for x in weights[i])
            f.write(f"{w} {vec}\n")
    return gensim.models.KeyedVectors.load_word2vec_format(path, binary=False)

kv_random   = model_to_keyedvectors(model_random,   vocab, id_to_word, './kv_random.txt')   if model_random   else None
kv_frozen   = model_to_keyedvectors(model_frozen,   vocab, id_to_word, './kv_frozen.txt')   if model_frozen   else None
kv_finetune = model_to_keyedvectors(model_ft,       vocab, id_to_word, './kv_finetune.txt') if model_ft       else None
print("KeyedVectors built for all three variants.")

In [ ]:
def show_neighbors(label, kv, word, topn=5):
    if kv is None or word not in kv:
        print(f"[{label}] '{word}' not in vocab -- skipping")
        return
    print(f"[{label}] most_similar('{word}')")
    for w, s in kv.most_similar(positive=[word], topn=topn):
        print(f"   {w:<20s} sim={s:.3f}")
    print()

for word in ['cheap', 'pizza', 'service']:
    print("=" * 60)
    show_neighbors('A: random  ', kv_random,   word)
    show_neighbors('B: frozen  ', kv_frozen,   word)
    show_neighbors('C: finetune', kv_finetune, word)

### When to freeze, when to fine-tune?

| Situation | Strategy |
|-----------|----------|
| Small dataset (<10K examples) | **Freeze**. You don't have enough signal to safely move the vectors; you'd just overfit. |
| Generic task (sentiment, news topics) | **Freeze** is often enough. The GloVe meaning is exactly what you want. |
| Domain-specific dataset (medical, legal, finance, restaurant reviews) | **Fine-tune** — domain words shift in meaning ("positive" in a medical chart vs. a movie review). |
| Very large dataset (>1M examples) | **Fine-tune** — you have enough data to safely re-learn even rare words. |

## Section 5: Evaluating Embedding Quality

Loss curves are nice but not satisfying. Let's evaluate the embeddings **directly**.

### Intrinsic evaluation: analogy accuracy

Word analogies are the classic intrinsic evaluation: given `a : b :: c : ?`, find `?` by computing `b - a + c` and looking at the nearest neighbor. We'll use a small built-in analogy set covering country–capital, gender, and verb-tense relations.

Expected ranking (ballpark — your numbers will vary slightly):

- **GloVe alone** (full 400K vocab): ~80%+ on capitals.
- **C: fine-tuned**: ~30–50% — vectors specialized to Yelp, but most analogies need words Yelp barely uses.
- **B: frozen**: ~25–45% — same starting GloVe, but Yelp vocab subset misses many analogy words.
- **A: random**: ~0% — vectors are basically noise for analogy tasks.

In [ ]:
# A small built-in analogy set (a : b :: c : d).
# We'll only score analogies whose 4 words ALL exist in the variant's vocab,
# so the comparison is apples-to-apples.
ANALOGIES = [
    # capitals
    ('paris', 'france', 'rome', 'italy'),
    ('paris', 'france', 'berlin', 'germany'),
    ('paris', 'france', 'madrid', 'spain'),
    ('paris', 'france', 'tokyo', 'japan'),
    ('paris', 'france', 'london', 'england'),
    # gender
    ('king',  'queen',  'man',   'woman'),
    ('boy',   'girl',   'father','mother'),
    ('boy',   'girl',   'son',   'daughter'),
    # verb tense
    ('go',    'went',   'do',    'did'),
    ('go',    'went',   'see',   'saw'),
]

def analogy_accuracy(kv, analogies, topn=5):
    correct, total = 0, 0
    for a, b, c, d in analogies:
        if any(w not in kv for w in (a, b, c, d)):
            continue
        # b - a + c should be near d
        try:
            preds = kv.most_similar(positive=[b, c], negative=[a], topn=topn)
        except KeyError:
            continue
        if d in [w for w, _ in preds]:
            correct += 1
        total += 1
    return correct, total

for label, kv in [('A: random', kv_random), ('B: frozen', kv_frozen), ('C: finetune', kv_finetune)]:
    if kv is None:
        continue
    c, t = analogy_accuracy(kv, ANALOGIES, topn=5)
    pct = 100 * c / max(1, t)
    print(f"{label:>12s} | analogy@top5 = {c}/{t}  ({pct:.0f}%)")

### Demo: t-SNE visualization

Project the top-300 most-frequent Yelp words to 2D with t-SNE and color by k-means cluster. Repeat for each variant. Frozen GloVe should look the most coherent (clusters that group semantically — *food*, *time*, *places*); random should look chaotic; fine-tuned shifts the GloVe layout toward Yelp-specific groupings.

In [ ]:
def tsne_plot(kv, title, n_words=300, n_clusters=5, ax=None):
    if kv is None:
        return
    # Pick the n_words most frequent words from our Yelp vocab that are also in this kv
    words = [w for w, _ in counter.most_common() if w in kv][:n_words]
    vectors = np.stack([kv[w] for w in words])
    # K-means cluster IDs for color
    km = KMeans(n_clusters=n_clusters, random_state=SEED, n_init=10).fit(vectors)
    # 2D projection
    coords = TSNE(n_components=2, random_state=SEED, perplexity=30,
                  init='pca', learning_rate='auto').fit_transform(vectors)
    if ax is None:
        ax = plt.gca()
    ax.scatter(coords[:, 0], coords[:, 1], c=km.labels_, cmap='tab10', s=18, alpha=0.7)
    ax.set_title(title); ax.set_xticks([]); ax.set_yticks([])

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
tsne_plot(kv_random,   'A: random init',     ax=axes[0])
tsne_plot(kv_frozen,   'B: frozen GloVe',    ax=axes[1])
tsne_plot(kv_finetune, 'C: fine-tuned GloVe', ax=axes[2])
plt.tight_layout(); plt.show()

### Lab 5.1: Compare neighbors of a domain word and a generic word

Pick two words: one **domain-specific** (something a restaurant reviewer would say — e.g. `'pizza'`, `'service'`, `'waiter'`, `'menu'`) and one **generic** (something Wikipedia would talk about — e.g. `'vehicle'`, `'science'`, `'history'`).

Print the top-5 nearest neighbors from each of the three KeyedVectors. You should see:

- For the **domain word**: fine-tuned > frozen > random. Yelp-specific neighbors emerge.
- For the **generic word**: frozen > fine-tuned > random. The fine-tuned model has *forgotten* a bit of GloVe's general-purpose knowledge in exchange for Yelp specialization. This is **catastrophic forgetting** in miniature.

Hint: use the `show_neighbors` helper from earlier.

In [ ]:
# Lab 5.1: compare a domain word and a generic word across variants
domain_word  = None  # YOUR CODE -- e.g. 'pizza' or 'service'
generic_word = None  # YOUR CODE -- e.g. 'vehicle' or 'science'

if domain_word and generic_word:
    print(f"\n===== DOMAIN WORD: '{domain_word}' =====")
    show_neighbors('A: random  ', kv_random,   domain_word)
    show_neighbors('B: frozen  ', kv_frozen,   domain_word)
    show_neighbors('C: finetune', kv_finetune, domain_word)

    print(f"\n===== GENERIC WORD: '{generic_word}' =====")
    show_neighbors('A: random  ', kv_random,   generic_word)
    show_neighbors('B: frozen  ', kv_frozen,   generic_word)
    show_neighbors('C: finetune', kv_finetune, generic_word)

## Section 6: Wrap-up

You just ran the canonical transfer-learning recipe for word embeddings:

1. Download a pretrained checkpoint (GloVe / Word2Vec / fastText / ...).
2. Build an aligned `(vocab_size, embedding_dim)` matrix where each row is the pretrained vector for that vocab word (with sensible OOV fallback).
3. Plug it into `nn.Embedding.from_pretrained(matrix, freeze=True/False, padding_idx=0)`.
4. Decide: freeze (safe, small data) or fine-tune (powerful, requires more data and a smaller LR).

**This is the pattern you will use in every downstream notebook**:

- **NB7** — sentence-transformers (already pretrained sentence vectors).
- **NB8** — feed pretrained word vectors into an MLP for news classification.
- **NB9** — BERT, where the entire transformer is the pretrained checkpoint.
- **NB11** — load embeddings into an LSTM language model.

### Self-check quiz

1. You have a domain corpus of 200 documents and want embeddings. Train from scratch, frozen GloVe, or fine-tuned GloVe? Defend your choice.
2. Why does `padding_idx=0` require row 0 of the embedding matrix to be all zeros?
3. After fine-tuning on Yelp, `most_similar('cheap')` now returns `['crappy', 'stale', 'mediocre']` instead of `['inexpensive', 'affordable']`. Is this a bug? Explain.
4. You loaded GloVe 50d but kept `EMBEDDING_DIM = 100` in the hyperparameters cell. What error will you see, and at which line?
5. For the **frozen** variant, why is it wasteful to pass `model.parameters()` to Adam without a `filter(lambda p: p.requires_grad, ...)`?

### Optional / extra labs

1. Try **GloVe 6B 50d** instead of 100d. Update `EMBEDDING_DIM` and the download URL accordingly. Does the analogy accuracy drop a lot?
2. Train a **4th variant**: random init but with the same LR as fine-tune (`5e-4`). Does the lower LR help or hurt the random model?
3. Fine-tune for **20 epochs** instead of 5. Does `most_similar('vehicle')` get worse over time as Yelp specialization deepens?

## Congratulations!

You can now do **transfer learning for embeddings** — the foundation for every modern NLP system. On to Notebook 7, where we move up one level of abstraction: from word vectors to **sentence vectors** with sentence-transformers.